In [17]:
!pip uninstall -y numpy scikit-surprise -q

In [2]:
!pip install numpy==1.26.4 --no-cache-dir
!pip install scikit-surprise==1.1.3 --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.0/772.0 kB 33.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.3-cp312-cp312-linux_x86_64.whl size=3513898 sha256=d85377097217f5ba0cc62167ffe6f73a102b361cd58a52109f0e750112ff6d73
  Stored in directory: /tmp/pip-ephem-wheel-cache-niqpt9vo/wheels/ee/08/67/4176eedbed1c63c15db21a526f1893ca43ee8453182a239afc
Successfully built scikit-surprise


In [16]:
import numpy as np
import surprise

print(np.__version__)
print("Surprise OK")

1.26.4
Surprise OK


In [17]:
%%writefile app.py

import streamlit as st
import pandas as pd
import pickle
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

st.set_page_config(page_title="Hybrid Recommender", page_icon="🎬")

st.title("🎬 Hybrid Movie Recommendation System")

st.write("Rate 3 movies and get recommendations.")

@st.cache_data
def load_data():

    movies = pd.read_csv("clean_content.csv")

    ratings = pd.read_csv(
        "ratings_title.csv",
        engine="python",
        on_bad_lines="skip"
    )

    ratings.rename(
        columns={"userId": "user_id", "movieId": "movie_id"},
        inplace=True
    )

    # ✅ CONTENT-BASED (TF-IDF)
    movies["genres"] = movies["genres"].fillna("")

    tfidf = TfidfVectorizer(stop_words="english")
    tfidf_matrix = tfidf.fit_transform(movies["genres"])

    similarity = cosine_similarity(tfidf_matrix)

    similarity_df = pd.DataFrame(
        similarity,
        index=movies["title"].values,
        columns=movies["title"].values
    )

    # ✅ SVD MODEL
    svd_model = pickle.load(open("svd_model.pkl", "rb"))

    return movies, ratings, similarity_df, svd_model


movies, ratings_base, similarity_df, svd_model = load_data()

st.subheader("Rate Movies")

movie_options = sorted(movies["title"].dropna().unique())

movie1 = st.selectbox("Movie 1", movie_options)
rate1 = st.slider("Rating 1", 1.0, 5.0, 4.0, 0.5)

movie2 = st.selectbox("Movie 2", movie_options, index=1)
rate2 = st.slider("Rating 2", 1.0, 5.0, 4.0, 0.5)

movie3 = st.selectbox("Movie 3", movie_options, index=2)
rate3 = st.slider("Rating 3", 1.0, 5.0, 4.0, 0.5)

user_input = [(movie1, rate1), (movie2, rate2), (movie3, rate3)]


def recommend():

    new_user_id = ratings_base["user_id"].max() + 1

    rows = []

    for title, rating in user_input:

        row = movies[movies["title"] == title]

        if row.empty:
            continue

        rows.append({
            "user_id": new_user_id,
            "movie_id": int(row["movieId"].values[0]),
            "rating": rating,
            "title": title
        })

    df = pd.concat([ratings_base, pd.DataFrame(rows)], ignore_index=True)

    # ---------- CONTENT ----------
    watched = [m for m, _ in user_input]

    content_scores = {}

    for movie in watched:

        if movie not in similarity_df.columns:
            continue

        sims = similarity_df[movie]

        for title, score in sims.items():

            if title in watched:
                continue

            content_scores[title] = content_scores.get(title, 0) + score

    cb_df = pd.DataFrame(
        content_scores.items(),
        columns=["title", "content_score"]
    )

    # ---------- COLLAB FILTER ----------
    user_mean = df.groupby("user_id")["rating"].mean()

    cf_scores = {}

    for _, row in df.iterrows():

        if row["title"] in watched:
            continue

        cf_scores[row["title"]] = cf_scores.get(row["title"], 0) + row["rating"]

    cf_df = pd.DataFrame(
        cf_scores.items(),
        columns=["title", "cf_score"]
    )

    # ---------- HYBRID ----------
    hybrid = pd.merge(cb_df, cf_df, on="title", how="inner")

    if hybrid.empty:
        return pd.DataFrame()

    hybrid["svd"] = hybrid["title"].apply(
        lambda x: svd_model.predict(
            new_user_id,
            int(movies[movies["title"] == x]["movieId"].values[0])
        ).est
    )

    hybrid["score"] = (
        0.4 * hybrid["content_score"] +
        0.3 * hybrid["cf_score"] +
        0.3 * hybrid["svd"]
    )

    hybrid = hybrid.sort_values("score", ascending=False).head(10)

    result = pd.merge(
        hybrid,
        movies[["title", "genres", "year"]],
        on="title"
    )

    return result


if st.button("🎯 Recommend"):

    with st.spinner("Generating..."):

        recs = recommend()

    if recs.empty:
        st.warning("Not enough data.")
    else:
        st.dataframe(recs)

Overwriting app.py


In [1]:
from google.colab import files
uploaded = files.upload()

Saving svd_model.pkl to svd_model.pkl
Saving ratings_title.csv to ratings_title.csv
Saving clean_content.csv to clean_content.csv


In [18]:
from pyngrok import ngrok

In [19]:
!ngrok config add-authtoken 3DdBofi6PoOVIyOSDgzd8hvEgDz_6ixXKwfky1upzLfnYND5B

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [20]:
!nohup streamlit run app.py --server.port 8501 &

nohup: appending output to 'nohup.out'


In [21]:
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://unwieldy-canon-morphing.ngrok-free.dev" -> "http://localhost:8501"
